In [1]:
!pip install transformers
!pip install torch

In [2]:
from transformers import pipeline

In [3]:
sentiment_pipeline = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [4]:
result = sentiment_pipeline("I love this project")
print(result)

[{'label': 'POSITIVE', 'score': 0.999884843826294}]


Build Sentiment Chatbot Logic

In [5]:
def chatbot():
    print("Sentiment Chatbot 🤖 (type 'exit' to stop)")

    while True:
        user_input = input("You: ")

        if user_input.lower() == "exit":
            print("Bot: Goodbye 👋")
            break

        # Get sentiment
        result = sentiment_pipeline(user_input)[0]
        label = result['label']

        # Bot response based on sentiment
        if label == "POSITIVE":
            print("Bot: That's great to hear 😊")
        else:
            print("Bot: I'm sorry to hear that 😔")

In [7]:
chatbot()

Sentiment Chatbot 🤖 (type 'exit' to stop)
You: I am sad
Bot: I'm sorry to hear that 😔
You: I am happy today
Bot: That's great to hear 😊
You: exit
Bot: Goodbye 👋


Convert into API

In [8]:
!pip install fastapi uvicorn nest-asyncio

In [14]:
!pip install requests

In [1]:
import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI
from transformers import pipeline
import uvicorn
import threading
import time

# Load model
sentiment_pipeline = pipeline("sentiment-analysis")

# Create app
app = FastAPI()

@app.get("/")
def home():
    return {"message": "API Running"}

@app.get("/predict")
def predict(text: str):
    result = sentiment_pipeline(text)[0]
    return {
        "input": text,
        "sentiment": result['label'],
        "score": result['score']
    }

# Run server in background
def run():
    uvicorn.run(app, host="127.0.0.1", port=8000)

thread = threading.Thread(target=run, daemon=True)
thread.start()

# ✅ WAIT FOR SERVER TO START
time.sleep(5)

print("✅ Server started")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/uvicorn/server.py:75: RuntimeWarning: coroutine 'Server.serve' was never awaited
  return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
Exception in thread Thread-4 (run):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_11628/2135011761.py", line 31, in run
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/main.py", line 606, in run
    server.run()
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 75, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: _patch_asyncio.<locals>.run() got an unexpected keyword argument 'l

✅ Server started


In [4]:
# Simulate API call directly

def test_api(text):
    result = sentiment_pipeline(text)[0]
    return {
        "input": text,
        "sentiment": result['label'],
        "score": result['score']
    }

# Test
response = test_api("I love AI")
print(response)

{'input': 'I love AI', 'sentiment': 'POSITIVE', 'score': 0.9998302459716797}
